# Selection of Representative Network Partition for Visualization

After repeating the experiment multiple times—partitioning the network and assigning roles in each iteration—we aim to identify the repetition that best represents the average behavior observed across all runs. This representative partition will be used for subsequent visualizations and analysis.

To achieve this, we compare each repetition to the consensus (the most frequently assigned roles for each node) using the Hamming distance. The repetition with the smallest Hamming distance to the consensus is selected as the most representative, ensuring that the chosen result closely matches the overall trends in the data.

In [3]:
import sys
sys.path.append('../bin')

import networkx as nx

import pandas as pd
import numpy as np
from scipy.spatial.distance import hamming
import random

g_files = ["no_carnival_tourist","carnival_tourist","no_carnival_residents","carnival_residents"]

## Calculate Hamming Distance to Consensus

For each repetition (each column in the DataFrame, representing a Louvain + Guimera execution), we calculate the Hamming distance to the consensus roles (the most frequently assigned role for each node). The repetition with the smallest Hamming distance is selected as the most representative. If there is a tie, one of the tied columns is chosen at random.

In [5]:
for n_file in g_files:
    
    df_community = pd.read_csv(f"../models/partial_results/{n_file}_guimera_repeated_cm.csv", index_col=0)
    df_z = pd.read_csv(f"../models/partial_results/{n_file}_guimera_repeated_z.csv", index_col=0)
    df_P = pd.read_csv(f"../models/partial_results/{n_file}_guimera_repeated_p.csv", index_col=0)

    df_role = pd.read_csv(f"../models/partial_results/{n_file}_guimera_repeated_role.csv", index_col=0)
    # df_reference_role = pd.read_csv(f"../models/{n_file}_guimera_repeated_role_mean.csv", index_col=0)
    df_reference_role = pd.read_csv(f"../models/partial_results/{n_file}_guimera_repeated_role_majority_vote.csv")
    
    # Skip the first column (identifiers)
    reference = df_reference_role.iloc[:, 1].values
    df_role_values = df_role.values

    cols = df_role.columns
    dist_list = []

    for i in range(len(cols)):
        dist = hamming(df_role_values[:, i], reference)
        dist_list.append(dist)

    # Create a DataFrame of results
    results = pd.DataFrame({
        'Column': cols,
        'distance_to_reference': dist_list
    }).sort_values('distance_to_reference')

    # Select the column most similar to the reference
    min_distance = results['distance_to_reference'].min()
    min_cols = results[results['distance_to_reference'] == min_distance]['Column'].tolist()
    selected_column = random.choice(min_cols)

    print(f"Hamming distances to reference (for {n_file}):")
    print(results)
    print("\nColumn most similar to the reference selected:")
    print(selected_column)

    # Save the selected community membership column as the final community membership
    # It is the one which most closesly resembles the reference roles calculated as the mean over repetitions
    df_community = df_community[[selected_column]]
    df_community.to_csv(f"../models/{n_file}_guimera_repeated_cm_final.csv")

    # Save the selected z and P columns as the final z and P
    df_z = df_z[[selected_column]]
    df_z.to_csv(f"../models/{n_file}_guimera_repeated_z_final.csv")

    df_P = df_P[[selected_column]]
    df_P.to_csv(f"../models/{n_file}_guimera_repeated_P_final.csv")

    # Save the selected role column as the final role
    df_role = df_role[[selected_column]]
    df_role.to_csv(f"../models/{n_file}_guimera_repeated_role_final.csv")

Hamming distances to reference (for no_carnival_tourist):
     Column  distance_to_reference
0  repeat_0               0.114754
1  repeat_1               0.114754
2  repeat_2               0.114754
3  repeat_3               0.114754
4  repeat_4               0.114754
5  repeat_5               0.114754
6  repeat_6               0.114754
7  repeat_7               0.114754
8  repeat_8               0.114754
9  repeat_9               0.262295

Column most similar to the reference selected:
repeat_8
Hamming distances to reference (for carnival_tourist):
     Column  distance_to_reference
0  repeat_0               0.052632
3  repeat_3               0.052632
5  repeat_5               0.052632
8  repeat_8               0.052632
2  repeat_2               0.245614
4  repeat_4               0.245614
9  repeat_9               0.245614
1  repeat_1               0.298246
6  repeat_6               0.298246
7  repeat_7               0.315789

Column most similar to the reference selected:
repeat_0
Ham